# Solution: Generating Microstructures

In [ ]:
import os
from urllib.request import urlretrieve
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import numpy as np
import torch

from torch import nn
from torch.nn import functional
from torch.utils.data import DataLoader, random_split

from typing import List, Callable, Union, Any, TypeVar, Tuple

from cycler import cycler
import seaborn as sns

# Set the color scheme
sns.set_theme()
colors = [
    "#0076C2",
    "#EC6842",
    "#A50034",
    "#009B77",
    "#FFB81C",
    "#E03C31",
    "#6CC24A",
    "#EF60A3",
    "#0C2340",
    "#00B8C8",
    "#6F1D77",
]

torch.set_default_tensor_type(torch.DoubleTensor)
# Tensor = TypeVar("torch.tensor")

## Introduction

In this exercise, we will look into using a Variational Autoencoder to find the hidden patterns that explain the geometry of complex microstructures. In the future, we should be able to design new metamaterials with highly efficient microscopic structures that will be specific to their envisioned application. For instance, the microstructures of materials used for different parts of an airplane could be radically different since different parts are subject to different loading conditions:

<center><img src="https://surfdrive.surf.nl/files/index.php/s/9uWO6YDuFRwuhoO/download"/></center>

In this notebook, we build a machine learning model to represent these microstructures with a very small number of latent dimensions. Once we find this representation, we can then sample from it and generate **new microstructures**, which can give us new ideas for structural design. The generative model we will be using here is related to other image generators, such as DALL-E.

### The Cahn-Hilliard dataset

We will be using images of microstructures generated with the Cahn-Hilliard PDE. We can already load the dataset, set up our `DataLoader`, and look at some microstructures:

In [ ]:
# set rng
torch.manual_seed(0)
g = torch.Generator()
g.manual_seed(0)

# Download the Cahn-Hilliard dataset (if necessary)
url = "https://surfdrive.surf.nl/s/Jis2zsxZSigqJod/download"
filename = "25x25.dat"

if not os.path.isfile(filename):
    print(f"Downloading {filename}...")
    urlretrieve(url, filename)

# set params for dataset handling
n_data = 30000
batch_size = 40
validation_frac = 0.7

# split dataset into training and validation sets
data_all = torch.tensor(np.loadtxt(filename)[:n_data], dtype=torch.float64)
# data_all = torch.tensor(np.loadtxt(filename)[:n_data], dtype=torch.float32)
data_train, data_val = random_split(
    data_all, [validation_frac, 1 - validation_frac], generator=g
)

training_loader = DataLoader(
    data_train, batch_size=batch_size, shuffle=True, generator=g
)
validation_loader = DataLoader(
    data_val, batch_size=batch_size, shuffle=True, generator=g
)

# plot a few samples from the training dataset
n_samples = 5
idcs = np.random.permutation(training_loader.dataset.indices)[:n_samples]
samples = training_loader.dataset.dataset[idcs]

fig, ax = plt.subplots(1, n_samples, figsize=(n_samples * 2.5, 2.5))

for i in range(n_samples):
    ax[i].imshow(samples[i].reshape(25, 25))

# adapt layout
[axs.set_axis_off() for axs in ax.flat]

plt.show()

## Trying it with PPCA

First let us try to use PPCA to find low-dimensional patterns in this dataset. We give you some code below that implements PPCA with the EM algorithm:

In [ ]:
# This version of EM looks different than the one we have seen in the lecture.
# It gives equivalent solutions but is much faster to run, so we opt for it here

def ppca_em(x,M, maxiter=20):
    N = x.shape[0]
    D = x.shape[1]

    # make sure the latent space is lower-dimensional than the original
    M = min(M,D)

    # initial values
    W = torch.rand(D, M)
    sig2 = torch.zeros(1)

    # Sample mean and covariance
    mu = torch.mean(x, axis=0)
    S = torch.cov(x.T)
    
    for i in range(maxiter):
        # E and M steps
        Minv = torch.linalg.inv(W.T @ W + sig2 * torch.eye(M))
        SW = S @ W
        Wnew = SW @ torch.linalg.inv(sig2 * torch.eye(M) + Minv @ W.T @ SW)
        sig2new = torch.trace(S - SW @ Minv @ Wnew.T) / float(D)

        # Prepare for the next iteration
        W = Wnew
        sig2 = sig2new  

    return W, mu, sig2

Now use PPCA to compress the images, starting with latent dimensionality $M=10$. **Play with the code below**, trying different values of $M$. How high should $M$ be for a good reconstruction to be obtained?

In [ ]:
torch.manual_seed(42)

# number of PCA components. --> play with this! <--
M = 10

# get all training images
x = data_train[:]

# compute ppca
W_ml, mu_ml, sig2_ml = ppca_em(x, M, maxiter=10)

# plot a few reconstructions from the validation dataset
K = 5
fig, ax = plt.subplots(K, 2, figsize=(5, 2.5 * K))

x_val = data_val[:K]

z = np.zeros((K, M))

# compute latent representation of image and convert it back to original space
for i in range(K):
    ax[i, 0].imshow(x_val[i].reshape(25, 25))
    z = torch.linalg.inv(W_ml.T @ W_ml + sig2_ml*torch.eye(M)) @ W_ml.T @ (x_val[i] - mu_ml)
    x_s = torch.matmul(W_ml, z) + mu_ml
    ax[i, 1].imshow(x_s.reshape(25, 25))

[axs.set_axis_off() for axs in ax.flat]
ax[0, 0].set_title("full image")
ax[0, 1].set_title("ppca mean")

plt.show()

It seems we need to go quite high, which is not a surprise. This dataset is too complex in its original 625-dimensional space, so finding a hyperplane (linear) that contains a lot of the data is extremely difficult unless we use a very high dimensional one.

But now let's try to improve our reconstructions by **allowing the latent mappings to be nonlinear**. We can do that with a Variational Autoencoder.

## Going for a Variational Autoencoder

The probabilistic principal component analysis from the previous notebook is a linear latent-variable model unsuitable for non-gaussian datasets.
The variational autoencoder (VAE) is a deep latent-variable model (DLVM) whose distributions are parameterized by deep neural networks.
A DLVM defines a joint distribution $p_{\boldsymbol{\theta}}(\boldsymbol{x},\boldsymbol{z})$ over the observed and latent variables $\boldsymbol{x}$ and $\boldsymbol{y}$. A standard factorization of the DLVM follows the structure

$$
p_{\boldsymbol{\theta}}(\boldsymbol{x},\boldsymbol{z}) = p_{\boldsymbol{\theta}}(\boldsymbol{z}) p_{\boldsymbol{\theta}}(\boldsymbol{x}|\boldsymbol{z}),
$$

where $p_{\boldsymbol{\theta}}(\boldsymbol{z})$ is often called *prior* distribution since it is not conditioned on any observations, and $p_{\boldsymbol{\theta}}(\boldsymbol{x}|\boldsymbol{z})$ is the *likelihood* of observing $\boldsymbol{x}$ given $\boldsymbol{z}$. Specifying these two distributions, the joint distribution is fully defined.
The marginal distribution of the observed variable $p_{\boldsymbol{\theta}}(\boldsymbol{x})$ is given by

$$
p_{\boldsymbol{\theta}}(\boldsymbol{x}) = \int p_{\boldsymbol{\theta}}(\boldsymbol{x}, \boldsymbol{z}) d\boldsymbol{z}.
$$

A great advantage of a DLVM is that the *marginal* $p_{\boldsymbol{\theta}}(\boldsymbol{x})$ can contain almost arbitrary dependencies, even if *prior* and *likelihood* are kept simple, e.g. following a normal distribution.
The statistical model can be inverted, giving rise to the *posterior* distribution given by

$$
p_{\boldsymbol{\theta}}(\boldsymbol{z}|\boldsymbol{x}) = \frac{{p_\boldsymbol{\theta}}(\boldsymbol{x}, \boldsymbol{z})}{p_{\boldsymbol{\theta}}(\boldsymbol{z})}.
$$

In practice, however, both the marginal and posterior distributions are *intractable*, i.e., we cannot analytically solve the integral in their expressions.
Therefore, we must approximate these quantities. The variational autoencoder model tackles this problem with *variational inference*.
A parametric *inference model* $q_{\boldsymbol{\phi}}(\boldsymbol{z}|\boldsymbol{x})$, also referred to as *encoder* or *recognition model* is introduced. We optimize the *variational parameters* $\boldsymbol{\phi}$ such that

$$
q_{\boldsymbol{\phi}}(\boldsymbol{z}|\boldsymbol{x}) \approx p_{\boldsymbol{\theta}}(\boldsymbol{z}|\boldsymbol{x}).
$$

We define our inference model as a multivariate normal distribution, with its mean $\boldsymbol{\mu}$ and the log of its variance $\mathrm{log}\boldsymbol{\sigma}$ given by a neural network:

$$
\begin{align}
(\boldsymbol{\mu}, \mathrm{log}\boldsymbol{\sigma}) &= \mathrm{EncoderNN}_{\boldsymbol{\phi}}(\boldsymbol{x})\\
q_{\boldsymbol{\phi}}(\boldsymbol{z}|\boldsymbol{x}) &= \mathcal{N}(\boldsymbol{z}|\boldsymbol{\mu}, \mathrm{diag}(\boldsymbol{\sigma})).
\end{align}
$$

To optimize $\boldsymbol{\phi}$, we require some measure to quantify if our model distribution is close to the desired distribution. A common measure for the closeness of two distributions is the Kullback-Leibler divergence (KL-divergence), given by

$$
\begin{align}
D_{\mathrm{KL}}(q_{\boldsymbol{\phi}}(\boldsymbol{z}|\boldsymbol{x})\parallel p_{\boldsymbol{\theta}}(\boldsymbol{z}|\boldsymbol{x}))
&= \mathbb{E}_{q_{\boldsymbol\phi}} \left[ \mathrm{log} \left[ \frac{q_{\boldsymbol{\phi}}(\boldsymbol{z}|\boldsymbol{x})}
{p_{\boldsymbol{\theta}}(\boldsymbol{z}|\boldsymbol{x})} \right] \right]\\
&= \underbrace{\mathbb{E}_{q_{\boldsymbol\phi}} \left[ \mathrm{log} \left[ \frac{q_{\boldsymbol{\phi}}(\boldsymbol{z}|\boldsymbol{x})}
{p_{\boldsymbol{\theta}}(\boldsymbol{z},\boldsymbol{x})} \right] \right]}_{=\mathcal{L}_{\boldsymbol{\theta},\boldsymbol{\phi}}(\boldsymbol{x}) \,\, \mathrm{(ELBO)}}
 + \underbrace{\mathbb{E}_{q_{\boldsymbol\phi}} \left[ \mathrm{log} p_{\boldsymbol{\theta}}(\boldsymbol{x}) \right]}_{= \mathrm{log}p_{\boldsymbol{\theta}}(\boldsymbol{x})}.
\end{align}
$$

We cannot minimize the KL divergence directly, as it depends on the posterior distribution. However, if we minimize the first term on the right-hand side, we can indirectly minimize the KL divergence between the variational and the true posterior distribution. This term is often referred to as the evidence lower bound (ELBO), as it forms a lower bound for the model evidence $p_{\boldsymbol{\theta}}(\boldsymbol{x})$.

Your job is to complete the implementation of the VAE. Specifically, the missing parts are:
- The encoder's final layer yields the mean and variance of the latent space.
- The loss function (ELBO).
- The reparameterization trick that enables us to efficiently sample from $p_{\boldsymbol{\theta}}(\boldsymbol{z}|\boldsymbol{x})$ and to backpropagate the gradient of the loss function through the encoder.

**Hint** It can be helpful to write the ELBO as the sum of the KL-divergence from the variational posterior to the prior and the KL-divergence from the variational posterior to the likelihood. The first term then expresses how regular the variational posterior is, and the second one corresponds to the reconstruction error of the VAE.

## Defining the VAE model

Define a VAE model in the cells below by filling in the gaps in the code:

- For simplicity, we can assume the architecture is symmetric. That means you should add `hidden_num` hidden layers to each half of the model;
- Make sure all hidden layers are activated. We suggest the `nn.SELU()` activation for this problem, but feel free to try other ones;
- Add layers and activations to the provided `encoder` and `decoder` lists, which after being put through a `nn.Sequential()` object can be used easily in forward mode (e.g. `self.encoder(x)`);
- The `encode()` function needs to return the posterior moments `mu` and `log_var` separately. Feel free to already make the layers going from the last hidden layer of the encoder to `mu` and `log_var` detached from `self.encoder`. Alternatively, you can include them in `self.encoder` and then slice the two parts in `encode()`;
- When performing the reparametrization trick, do not forget to pass `log_var` through a `torch.exp`. This will guarantee that the learned `log_var` is a log-scaled version, and the exponentiation will naturally prevent negative variance components (which lead to improper Gaussians);
- For the loss function, make sure both loss terms are included and that the weighing parameter `beta` is taken into account. Check the online book for the expression of the loss.

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dim, hidden_num, beta=0.001):
        """
        Initializes the layers that make up the two halves of the autoencoder.

        Receives:
        'input_dim': Data dimensionality in the original space (D)
        'latent_dim': Dimensionality of the latent space z (M)
        'hidden_dim': Number of units of each hidden layer
        'hidden_num': Number of hidden layers of each half of the model
        'beta': The (optional) weighing of the KL divergence loss term

        Stores:
        'self.encoder': The encoder model, ready to be called
        'self.decoder': The decoder model, ready to be called
        """

        super(VAE, self).__init__()

        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.beta = beta

        # build the encoder
        encoder = []

        # TIP: encoder.append(nn.Linear(...))
        # ---------------------- student exercise --------------------------------- #
        encoder.append(nn.Linear(input_dim, hidden_dim))
        encoder.append(nn.SELU())

        for i in range(hidden_num):
            encoder.append(nn.Linear(hidden_dim, hidden_dim))
            encoder.append(nn.SELU())

        self.hidtomu = nn.Linear(hidden_dim, latent_dim)
        self.hidtovar = nn.Linear(hidden_dim, latent_dim)
        # ---------------------- student exercise --------------------------------- #

        self.encoder = nn.Sequential(*encoder)

        # build the decoder
        decoder = []

        # TIP: decoder.append(nn.Linear(...))
        # ---------------------- student exercise --------------------------------- #

        decoder.append(nn.Linear(latent_dim, hidden_dim))
        decoder.append(nn.SELU())

        for i in range(hidden_num):
            decoder.append(nn.Linear(hidden_dim, hidden_dim))
            decoder.append(nn.SELU())

        decoder.append(nn.Linear(hidden_dim, input_dim))
        decoder.append(nn.Sigmoid())

        # ---------------------- student exercise --------------------------------- #

        self.decoder = nn.Sequential(*decoder)

    def encode(self, x):
        """
        Encodes data from real space x to a Gaussian approximation q(z) of p(z|x)

        Receives:
        'x': a set of input features [B x D]

        Returns:
        'mu': Vector with posterior means of q(z) [B x M]
        'log_var': Vector with posterior covariances of q(z) [B x M]
        """

        # TIP: latents = self.encoder(...)
        # TIP: mu = ...
        # TIP: log_var = ...
        # ---------------------- student exercise --------------------------------- #
        result = self.encoder(x)

        mu = self.hidtomu(result)
        log_var = self.hidtovar(result)
        # ---------------------- student exercise --------------------------------- #
        return mu, log_var

    def decode(self, z):
        """
        Decodes data from latent space z to real space x

        Receives:
        'z': samples of q(z) of [B x M]

        Returns:
        'xtilde': decoded reconstructions (means of p(x|z)) [B x D]
        """

        result = self.decoder(z)

        return result

    def reparameterize(self, mu, logvar):
        """
        Performs the 'reparametrization trick' to draw a sample from q(z) while
        actually sampling N(0,I). The variance is exponentiated to learn a log-scaled
        version of it. This enforces non-negativity and ensures q(z) is a valid Gaussian.

        Receives:
        'mu': Mean of the latent Gaussian q(z) [B x M]
        'logvar': Log of (diagonal) variance of the latent Gaussian q(z) [B x M]

        Returns:
        'z': one sample of q(z) for each minibatch point [B x M]
        """

        # TIP: use torch.exp(...) and torch.randn_like(...)
        # ---------------------- student exercise --------------------------------- #
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = eps * std + mu
        # ---------------------- student exercise --------------------------------- #

        return z

    def forward(self, x):
        """
        Performs a complete forward pass through the autoencoder

        Receives:
        'x': A set of input features [B x D]

        Returns:
        'xtilde': A set of reconstructed inputs [B x D]
        'mu': The means of the posteriors q(z) [B x M]
        'log_var': The (naturally log-scaled) variances of the posteriors q(z) [B x M]
        """

        # TIP: encode, reparametrize and decode (check slides/book)
        # ---------------------- student exercise --------------------------------- #
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        xtilde = self.decode(z)
        # ---------------------- student exercise --------------------------------- #

        return xtilde, mu, log_var

    def generate(self, num_samples):
        """
        Samples from the prior p(z) and uses the decoder to generate new data

        Receives:
        'num_samples': number of samples to generate (integer)

        Returs:
        'samples': the samples in the original space [num_samples x D]
        """

        # TIP: generate samples from the prior and decode them all at once
        # TIP: use torch.randn(...)
        # ---------------------- student exercise --------------------------------- #
        z = torch.randn(num_samples, self.latent_dim)

        samples = self.decode(z)
        # ---------------------- student exercise --------------------------------- #

        return samples

    def loss_function(self, x, xtilde, mu, log_var):
        """
        Computes the loss function of the VAE
        """

        reconstruction_loss = functional.mse_loss(xtilde, x)

        # compute the KL divergence component of the loss
        # TIP: use torch.mean to average over the minibatch and a torch.sum over M
        # TIP: check the return statement for variable names you should include
        # ---------------------- student exercise --------------------------------- #
        kld_loss = torch.mean(
            -0.5 * torch.sum(1 + log_var - mu**2 - log_var.exp(), dim=1), dim=0
        )

        loss = reconstruction_loss + self.beta * kld_loss
        # ---------------------- student exercise --------------------------------- #

        return {
            "loss": loss,
            "recon": reconstruction_loss.detach(),
            "kld": kld_loss.detach(),
        }

## Auxiliary functions

Below we define a function for training the model for a full epoch, and some functions for plotting predictions. Note how we the inputs and targets are the same for this model.

In [ ]:
def train_one_epoch(loader):
    running_loss = 0.0
    last_loss = 0.0
    n_batch = loader.batch_size
    n_data = np.floor(loader.dataset.dataset.size()[0])

    write_every = int(np.floor(n_data / n_batch / 10))

    for i, data in enumerate(loader):
        x = data

        optimizer.zero_grad()

        xtilde, mu, log_var = model(x)

        output = model.loss_function(x, xtilde, mu, log_var)
        loss = output["loss"]
        loss.backward()

        optimizer.step()

        # periodically print current progress
        running_loss += loss.item()
        if i % write_every == write_every - 1:
            last_loss = running_loss / write_every  # loss per batch
            print(
                f"  loss: {output['loss']:5f}, "
                f"reconstruction loss: {output['recon']:5f}, "
                f"kld loss: {output['kld']:5f}"
            )
            running_loss = 0.0

    return last_loss


def plot_one_sample(loader):
    idcs = np.random.permutation(loader.dataset.indices)[:1]
    targets = loader.dataset.dataset[idcs]
    model.eval()
    recons = model.forward(targets)[0].detach()

    # create figure and plot reconstructions
    fig, ax = plt.subplots(1, 2, figsize=(2 * 2.5, 2.5))

    ax[0].imshow(targets[0].reshape(25, 25))
    ax[1].imshow(recons[0].reshape(25, 25))

    # adapt layout
    [axs.set_axis_off() for axs in ax.flat]
    ax[0].set_title("target")
    ax[1].set_title("reconstruction")

    plt.show()

## Initialize and train your model

Pick a size for the latent space, a value for beta and a network architecture and train a model to see what you get. The best model (with lowest validation loss) will be saved to a file automatically. If you would like to enable this feature, set `save_model = True`

In [ ]:
# model parameters
latent_dim = 10
kld_weight = 5e-04
hidden_dim = 100
hidden_num = 1

save_model = True

# create a folder to store trained models
if save_model:
    model_path = os.path.join(
        os.getcwd(), "models", f"lat-{latent_dim}_kld-{kld_weight:2.2e}"
    )
    os.makedirs(model_path, exist_ok=True)

# arrays to keep track of the losses
train_loss = []
val_loss = []

# create the model
model = VAE(
    input_dim=data_train.dataset.shape[1],
    latent_dim=latent_dim,
    hidden_dim=hidden_dim,
    hidden_num=hidden_num,
    beta=kld_weight,
)

# define optimizer
optimizer = torch.optim.Adam(model.parameters())
epoch_number = 0
best_vloss = 1_000_000.0
best_tloss = 1_000_000.0

# set numer of epochs and run training loop
epochs = 10

for epoch in range(epochs):
    print("--------------------------------------------")
    print("EPOCH {}:".format(epoch_number + 1))

    model.train(True)
    avg_loss = train_one_epoch(training_loader)

    running_vloss = 0.0
    model.eval()

    with torch.no_grad():
        for i, vdata in enumerate(validation_loader):
            x = vdata
            xtilde, mu, log_var = model(x)
            vloss = model.loss_function(x, xtilde, mu, log_var)["loss"]
            running_vloss += vloss

    avg_vloss = running_vloss / (i + 1)
    print(f"\nLOSS train {avg_loss:5f} valid {avg_vloss:5f}\n")
    train_loss.append(avg_loss)
    val_loss.append(avg_vloss)

    # track best validation performance, optionally save the model with lowest loss
    if avg_vloss < best_vloss:
        best_vloss = avg_vloss
        if save_model:
            torch.save(model.state_dict(), os.path.join(model_path, "model_best.pt"))

    epoch_number += 1

    # reconstruct one sample to give an idea of the progress
    plot_one_sample(validation_loader)

# plot complete training and validation loss history
fig, ax = plt.subplots()
ax.plot(train_loss, label="training loss")
ax.plot(val_loss, label="validation loss")
ax.set_ylabel("Loss")
ax.set_xlabel("Epochs")
ax.legend()
plt.show()

Set `epochs = 10` and try out different combinations of models:

- Single hidden layer (per half) of 10 units, 1 latent dimension, `kld_weight = 5e-04`
- Five hidden layers of 100 units, 1 latent dimension, `kld_weight = 5e-04`
- Five hidden layers of 100 units, 10 latent dimensions, `kld_weight = 1.0`
- Single hidden layer (per half) of 100 units, 20 latent dimensions, `kld_weight = 5e-04`

Do you see improvements in the reconstruction quality during these first epochs? Can you think of explanations for what you see? Does the quality of the reconstruction depend on the complexity of each microstructure?

With the intuition you gain from these trials, set `epochs = 100` and pick a model to train you think is the most promising. After training, keep going below and look at some generations.

## Generating new microstructures

Here we sample the latent space and pass it through the decoder to generate new, unseen, samples. Also experiment with setting `kld_weight=0` and seeing how it affects the generations we get here. Do you get generations with a good variety between samples?

In [ ]:
# load the best model, if needed
if save_model:
    model_path = os.path.join(
        os.getcwd(), "models", f"lat-{latent_dim}_kld-{kld_weight:2.2e}"
    )
    model_name = "model_best.pt"

    model = VAE(
        input_dim=data_train.dataset.shape[1],
        latent_dim=latent_dim,
        hidden_dim=hidden_dim,
        hidden_num=hidden_num,
        beta=kld_weight,
    )

    print(f"Attempting to load model ...")
    feedback = model.load_state_dict(torch.load(os.path.join(model_path, model_name)))
    print(f"  {feedback}")

# generate n samples (ancestral sampling)
n = 10
n_cols = 3
n_rows = int(n / n_cols) + 1
samples = model.generate(n).detach()
fig, ax = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.5))

# create figure and plot samples
for i in range(n):
    ax.flat[i].imshow(samples[i].reshape(25, 25))

[axs.set_axis_off() for axs in ax.flat]
[ax.flat[i].set_title(f"Sample {i + 1}") for i in range(n)]

plt.show()

## Wrapping up

So, we had images of realistic microstructures and we now can generate new ones with machine learning. Those could in principle be used to define all-new civil engineering materials.

But most important here is how well did the VAE work. Did it offer better performance than PCA at lower latent dimensionalities? How difficult is it to train and tweak? You now got a taste of how generative ML works!